# Домашнее задание 6.1. Перенос импульса по кольцу

**Уравнение:** $u_t+c u_x=0$, $x\in[0,1)$, $c=1$. Граничные условия периодические.

В работе сравниваются две явные разностные схемы: **upwind** (против потока) и **Лакса—Вендроффа**. Сначала проверим точность на синусоиде, затем посмотрим, что происходит с прямоугольным импульсом.

Обозначения: $N$ — число узлов, $\Delta x=1/N$, $C=c\Delta t/\Delta x$ — число Куранта. Точное решение получается сдвигом начального профиля:

$$u(x,t)=u_0\big((x-ct)\bmod 1\big).$$

Вычислительная сетка содержит точки $x_j=j/N$, $j=0,\dots,N-1$. Узел $x=1$ не добавляем, поскольку он совпадает с $x=0$.

## 1. Схемы и основные функции

Для $c>0$ схема upwind берёт разность против направления переноса:

$$u_j^{n+1}=u_j^n-C\,(u_j^n-u_{j-1}^n).$$

Для схемы Лакса—Вендроффа используем формулу

$$u_j^{n+1}=u_j^n-\frac{C}{2}(u_{j+1}^n-u_{j-1}^n)
+\frac{C^2}{2}(u_{j+1}^n-2u_j^n+u_{j-1}^n).$$

Индексы периодические: соседи крайнего узла находятся с другого конца массива. В NumPy это удобно записать через `np.roll`.

Будем вычислять два показателя:

$$E_1(t)=\Delta x\sum_j\left|u_j(t)-u_{\mathrm{exact}}(x_j,t)\right|,
\qquad M(t)=\Delta x\sum_j u_j(t).$$

При последнем шаге до заданного времени $t=1$ или $t=5$ уменьшаем $\Delta t$, чтобы попасть ровно в контрольный момент. Особенно для $N=50$, $C=0.8$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from IPython.display import display, Image

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.23, "axes.titlesize": 11, "legend.fontsize": 9,
    "lines.linewidth": 1.7,
})

COLORS = {"upwind": "#d97706", "lax_wendroff": "#2563eb", "exact": "#202020"}
N_VALUES = [50, 100, 200]

In [ ]:
def initial_profile(x, kind="smooth"):
    if kind == "smooth":
        return 1.0 + 0.5 * np.sin(2 * np.pi * x)
    if kind == "pulse":
        # Левая граница включена, правая — нет.
        z = np.round(np.asarray(x) % 1.0, 12) % 1.0
        return ((z >= 0.2) & (z < 0.4)).astype(float)
    raise ValueError("Неизвестный начальный профиль")


def exact_solution(x, t, kind="smooth"):
    # t % 1 помогает точно воспроизвести целое число оборотов.
    z = (np.asarray(x) - (t % 1.0)) % 1.0
    return initial_profile(z, kind)


def numerical_step(u, C, method):
    left = np.roll(u, 1)     # u[j-1] с периодическим переходом
    if method == "upwind":
        return u - C * (u - left)
    if method == "lax_wendroff":
        right = np.roll(u, -1)
        return u - 0.5 * C * (right - left) + 0.5 * C**2 * (right - 2*u + left)
    raise ValueError("Метод должен быть 'upwind' или 'lax_wendroff'")


def mass(u):
    return float(np.mean(u))   # dx * sum(u) = sum(u)/N


def l1_error(u, x, t, kind):
    return float(np.mean(np.abs(u - exact_solution(x, t, kind))))

Функция ниже ведёт расчёт до `t_end`. Параметр `checkpoints` задаёт моменты, в которые сохраняются копии профилей. Историю массы, ошибки, минимума и максимума записываем на каждом шаге, чтобы построить временные графики без повторного решения задачи.

In [ ]:
def solve(N, courant, t_end, kind, method, checkpoints=()):
    x = np.arange(N) / N
    u = initial_profile(x, kind).copy()
    dx = 1.0 / N
    dt_max = courant * dx       # c = 1
    t = 0.0

    times = [t]
    masses = [mass(u)]
    errors = [l1_error(u, x, t, kind)]
    minima = [float(u.min())]
    maxima = [float(u.max())]
    snapshots = {0.0: u.copy()}

    stops = sorted(set(float(s) for s in checkpoints if 0 < s < t_end)) + [float(t_end)]
    for stop in stops:
        while stop - t > 1e-12:
            dt = min(dt_max, stop - t)
            C_step = dt / dx
            u = numerical_step(u, C_step, method)
            t += dt

            times.append(t)
            masses.append(mass(u))
            errors.append(l1_error(u, x, t, kind))
            minima.append(float(u.min()))
            maxima.append(float(u.max()))

        t = stop
        snapshots[stop] = u.copy()

    return {
        "x": x, "snapshots": snapshots,
        "time": np.asarray(times),
        "mass": np.asarray(masses), "error": np.asarray(errors),
        "min": np.asarray(minima), "max": np.asarray(maxima),
    }


def get_row(result, N, method, t, kind):
    u = result["snapshots"][float(t)]
    return {"N": N, "метод": "Upwind" if method == "upwind" else "Лакс—Вендрофф",
            "t": t, "E1": l1_error(u, result["x"], t, kind),
            "M": mass(u), "min": u.min(), "max": u.max()}

## 2. Гладкий профиль: синусоида

$$u_0(x)=1+\frac12\sin(2\pi x),\qquad C=0.8.$$

Один оборот занимает $T=1$. После одного и пяти оборотов точный профиль совпадает с начальным, поэтому можно увидеть, насколько численное решение потеряло амплитуду или сдвинулось по фазе.

In [ ]:
METHODS = ["upwind", "lax_wendroff"]
SMOOTH_C = 0.8
smooth = {}
rows_smooth = []
for N in N_VALUES:
    for method in METHODS:
        result = solve(N, SMOOTH_C, 5.0, "smooth", method, checkpoints=(1.0,))
        smooth[(N, method)] = result
        for t in [1.0, 5.0]:
            rows_smooth.append(get_row(result, N, method, t, "smooth"))

table_smooth = pd.DataFrame(rows_smooth)
display(table_smooth[["N", "метод", "t", "E1", "M"]].style.format({"E1": "{:.7f}", "M": "{:.15f}"}))

In [ ]:
x_fine = np.linspace(0, 1, 1500)  # только для рисования, не для расчёта
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True, sharey=True)
for r, N in enumerate(N_VALUES):
    for c, t in enumerate([1.0, 5.0]):
        ax = axes[r, c]
        ax.plot(x_fine, exact_solution(x_fine, t), "--", color=COLORS["exact"], label="Точное")
        for method, label in [("upwind", "Upwind"), ("lax_wendroff", "Лакс—Вендрофф")]:
            s = smooth[(N, method)]
            ax.plot(s["x"], s["snapshots"][t], color=COLORS[method], label=label)
        ax.set_title(f"N={N}, t={t:g}")
        ax.set_xlim(0, 1)
        ax.set_ylabel("u")
        if r == 2:
            ax.set_xlabel("x")
        if r == 0 and c == 1:
            ax.legend(loc="lower left")
fig.suptitle("Синусоида: точное решение и две схемы", fontsize=14)
fig.tight_layout()
plt.show()

### Как уменьшается ошибка при сгущении сетки

На графике по горизонтали отложен шаг $\Delta x$, по вертикали — ошибка $E_1$. Обе оси логарифмические. Если $E_1\sim (\Delta x)^p$, наклон прямой приблизительно равен порядку $p$. Оценим его методом наименьших квадратов по трём сеткам.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
slopes = []
for ax, t in zip(axes, [1.0, 5.0]):
    h = np.array([1.0 / N for N in N_VALUES])
    for method, label in [("upwind", "Upwind"), ("lax_wendroff", "Лакс—Вендрофф")]:
        err = np.array([l1_error(smooth[(N, method)]["snapshots"][t],
                                 smooth[(N, method)]["x"], t, "smooth") for N in N_VALUES])
        p = np.polyfit(np.log(h), np.log(err), 1)[0]
        slopes.append({"t": t, "метод": label, "наблюдаемый порядок": p})
        ax.loglog(h, err, "o-", color=COLORS[method], label=f"{label}, p≈{p:.2f}")
    ax.set_title(f"После {t:g} оборота(ов)")
    ax.set_xlabel("Шаг сетки Δx")
    ax.set_ylabel("Ошибка E1")
    ax.legend()
fig.suptitle("Сходимость для гладкого профиля", fontsize=13)
fig.tight_layout()
plt.show()
display(pd.DataFrame(slopes).style.format({"наблюдаемый порядок": "{:.3f}"}))

### Проверка сохранения количества примеси

Возьмём $N=100$. Начальная масса синусоиды равна $M(0)=1$. Поскольку обе схемы записаны через разности соседних узлов, при суммировании по периодической сетке эти разности сокращаются. Поэтому обе схемы консервативны в точной арифметике, хотя распределение примеси они передают по-разному.

In [ ]:
N_mass = 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method, label in [("upwind", "Upwind"), ("lax_wendroff", "Лакс—Вендрофф")]:
    s = smooth[(N_mass, method)]
    axes[0].plot(s["time"], s["mass"], color=COLORS[method], label=label)
    axes[1].plot(s["time"], s["mass"] - s["mass"][0], color=COLORS[method], label=label)
axes[0].set(title="Полная масса M(t)", xlabel="t", ylabel="M")
axes[0].ticklabel_format(useOffset=False, axis="y")
axes[1].set(title="Отклонение от начальной массы", xlabel="t", ylabel="M(t) − M(0)")
axes[1].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()
print("Наибольшее |M(t) - M(0)| за 5 оборотов:")
for method in METHODS:
    s = smooth[(N_mass, method)]
    print(f"  {method:14s}: {np.max(np.abs(s['mass'] - s['mass'][0])):.3e}")

**Промежуточный вывод.** На гладкой функции upwind заметно уменьшает амплитуду (численная диффузия), особенно за пять оборотов. Лакс—Вендрофф точнее и даёт примерно второй порядок сходимости, но само сохранение массы ничего не говорит о точности формы решения. Следующий опыт это хорошо показывает.

## 3. Прямоугольный импульс

Теперь начальное условие имеет два разрыва:

$$u_0(x)=\begin{cases}1,&0.2\le x<0.4,\\ 0,&\text{иначе} .\end{cases}$$

При $N=200$ проверяем $C=0.5,\;0.8,\;1$. Отдельно рассматриваем $C=1.1$ — значение за пределом устойчивости обеих схем.

Для оценки паразитных колебаний нельзя обрезать график по диапазону $[0,1]$: отрицательные значения и превышения единицы — важная часть результата.

In [ ]:
N_PULSE = 200
STABLE_C = [0.5, 0.8, 1.0]
pulse = {}
rows_pulse = []
for C in STABLE_C + [1.1]:
    for method in METHODS:
        # Для неустойчивого режима дополнительно сохраняем промежуточные профили.
        checks = (0.15, 0.5) if C > 1.0 else ()
        s = solve(N_PULSE, C, 1.0, "pulse", method, checkpoints=checks)
        pulse[(C, method)] = s
        row = get_row(s, N_PULSE, method, 1.0, "pulse")
        row["C"] = C
        rows_pulse.append(row)

table_pulse = pd.DataFrame(rows_pulse)
display(table_pulse[["C", "метод", "E1", "min", "max", "M"]].style.format({
    "E1": "{:.4e}", "min": "{:.4e}", "max": "{:.4e}", "M": "{:.6e}"
}))

### Профили после одного оборота

Все три устойчивых режима показаны в одинаковом масштабе по вертикали. Пунктир — точный прямоугольник, оранжевая линия — upwind, синяя — Лакс—Вендрофф.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11.5, 9), sharex=True, sharey=True)
lo = min(0.0, *(pulse[(C, m)]["snapshots"][1.0].min() for C in STABLE_C for m in METHODS))
hi = max(1.0, *(pulse[(C, m)]["snapshots"][1.0].max() for C in STABLE_C for m in METHODS))
padding = 0.08 * (hi - lo)
full_ylim = (lo - padding, hi + padding)
for ax, C in zip(axes, STABLE_C):
    ax.plot(x_fine, exact_solution(x_fine, 1.0, "pulse"), "--",
            color=COLORS["exact"], label="Точное", linewidth=1.7)
    for method, label in [("upwind", "Upwind"), ("lax_wendroff", "Лакс—Вендрофф")]:
        s = pulse[(C, method)]
        ax.plot(s["x"], s["snapshots"][1.0], color=COLORS[method], label=label)
    ax.set(title=f"C = {C:g},  t = 1", ylabel="u", ylim=full_ylim, xlim=(0, 1))
    ax.legend(loc="upper right", ncol=3)
axes[-1].set_xlabel("x")
fig.suptitle("Прямоугольный импульс после полного оборота", fontsize=13)
fig.tight_layout()
plt.show()

### Увеличение обоих фронтов

Левый фронт находится около $x=0.2$, правый — около $x=0.4$. Используются **одинаковая ширина окна и общая шкала по вертикали** для всех шести рисунков. Масштаб учитывает реальные минимумы и максимумы расчётов, а не только допустимый физический диапазон.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharey=True)
fronts = [(0.2, "Левый фронт"), (0.4, "Правый фронт")]
for r, C in enumerate(STABLE_C):
    for col, (center, front_name) in enumerate(fronts):
        ax = axes[r, col]
        ax.plot(x_fine, exact_solution(x_fine, 1.0, "pulse"), "--", color=COLORS["exact"], label="Точное")
        for method, label in [("upwind", "Upwind"), ("lax_wendroff", "Лакс—Вендрофф")]:
            s = pulse[(C, method)]
            ax.plot(s["x"], s["snapshots"][1.0], color=COLORS[method], label=label)
        ax.set(xlim=(center-0.075, center+0.075), ylim=full_ylim,
               title=f"{front_name}, C={C:g}", xlabel="x", ylabel="u")
        if r == 0 and col == 1:
            ax.legend(loc="upper right")
fig.suptitle("Фронты: одинаковый масштаб для сравнения", fontsize=13)
fig.tight_layout()
plt.show()

### Как со временем меняются минимум, максимум и ошибка

Для устойчивых значений $C$ приводим отдельные графики по двум схемам. Ожидается, что upwind не создаёт новых экстремумов при $0\le C\le1$, а Лакс—Вендрофф может давать значения ниже нуля и выше единицы возле разрывов.

In [ ]:
metric_info = [("min", "Минимум u", 0.0),
               ("max", "Максимум u", 1.0),
               ("error", "Ошибка E1", 0.0)]
c_colors = {0.5: "#16a34a", 0.8: "#7c3aed", 1.0: "#dc2626"}
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for col, method in enumerate(METHODS):
    for row, (metric, title, reference) in enumerate(metric_info):
        ax = axes[row, col]
        for C in STABLE_C:
            s = pulse[(C, method)]
            ax.plot(s["time"], s[metric], color=c_colors[C], label=f"C={C:g}")
        ax.axhline(reference, color="#333333", linestyle="--", linewidth=1, alpha=0.5)
        ax.set(title=("Upwind" if method == "upwind" else "Лакс—Вендрофф") + " — " + title,
               ylabel=title, xlim=(0, 1))
        if row == 2:
            ax.set_xlabel("Время t")
        if row == 0:
            ax.legend()
fig.suptitle("Устойчивые расчёты (C≤1), N=200", fontsize=13)
fig.tight_layout()
plt.show()

## 4. Что меняется при $C=1.1$

Проверка устойчивости для обеих схем даёт условие $0\le C\le1$. При $C>1$ отдельные гармоники начинают усиливаться на каждом шаге. Из-за этого разностное решение перестаёт быть ограниченным.

**Эти графики отдельно от устойчивых расчётов.** На временных графиках экстремумов применена симметричная логарифмическая шкала (`symlog`), поэтому сохраняются и отрицательные значения, и большие положительные выбросы. Масштаб нигде не ограничен вручную диапазоном $[0,1]$.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for col, method in enumerate(METHODS):
    s = pulse[(1.1, method)]
    for row, (metric, title, reference) in enumerate(metric_info):
        ax = axes[row, col]
        ax.plot(s["time"], s[metric], color=COLORS[method])
        if metric == "error":
            ax.set_yscale("log")
            # Начальная ошибка нулевая: она не отображается на log-шкале.
        else:
            ax.axhline(reference, color="black", ls="--", lw=0.9, alpha=0.4)
            ax.set_yscale("symlog", linthresh=1e-2)
        ax.set(title=("Upwind" if method == "upwind" else "Лакс—Вендрофф") + " — " + title,
               ylabel=title, xlim=(0, 1))
        if row == 2:
            ax.set_xlabel("Время t")
fig.suptitle("Неустойчивые расчёты: C=1.1, N=200", fontsize=13)
fig.tight_layout()
plt.show()

Чтобы увидеть не только рост ошибки, но и саму форму решения, нарисуем профили в разные моменты времени. **Оси $u$ автоматически подстраиваются под все полученные значения:** ничего не обрезаем. По мере роста амплитуды точный импульс становится почти незаметным в таком масштабе — это и есть признак разрушения результата.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for row, t in enumerate([0.15, 0.5, 1.0]):
    for col, method in enumerate(METHODS):
        ax = axes[row, col]
        s = pulse[(1.1, method)]
        ax.plot(x_fine, exact_solution(x_fine, t, "pulse"), "--", color=COLORS["exact"], label="Точное")
        ax.plot(s["x"], s["snapshots"][t], color=COLORS[method], label="Численное")
        ax.set(title=f"{'Upwind' if method == 'upwind' else 'Лакс—Вендрофф'},  t={t:g}",
               ylabel="u", xlim=(0, 1))
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-3, 3))
        if row == 0:
            ax.legend(loc="upper right")
        if row == 2:
            ax.set_xlabel("x")
fig.suptitle("Развитие неустойчивости без ограничения экстремумов", fontsize=13)
fig.tight_layout()
plt.show()

## 5. Анимация переноса

Анимация показывает два оборота прямоугольного импульса для $N=200$, $C=0.8$. Сравниваются точное решение и обе схемы. Видно, как upwind постепенно размывает фронты, а у Лакса—Вендроффа рядом с разрывами возникают колебания.


In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path
import tempfile

N_anim, C_anim = 200, 0.8
x_anim = np.arange(N_anim) / N_anim
u_up = initial_profile(x_anim, "pulse").copy()
u_lw = u_up.copy()
dt_anim = C_anim / N_anim

# Два оборота: 500 шагов, кадр через каждые 10 шагов.
frames = [(0.0, u_up.copy(), u_lw.copy())]
for k in range(1, 501):
    u_up = numerical_step(u_up, C_anim, "upwind")
    u_lw = numerical_step(u_lw, C_anim, "lax_wendroff")
    if k % 10 == 0:
        frames.append((k * dt_anim, u_up.copy(), u_lw.copy()))

all_min = min(0.0, *(min(f[1].min(), f[2].min()) for f in frames))
all_max = max(1.0, *(max(f[1].max(), f[2].max()) for f in frames))
margin = 0.07 * (all_max - all_min)

fig, ax = plt.subplots(figsize=(9, 4.5))
line_exact, = ax.plot([], [], "--", color=COLORS["exact"], label="Точное")
line_up, = ax.plot([], [], color=COLORS["upwind"], label="Upwind")
line_lw, = ax.plot([], [], color=COLORS["lax_wendroff"], label="Лакс—Вендрофф")
ax.set(xlim=(0, 1), ylim=(all_min-margin, all_max+margin), xlabel="x", ylabel="u")
ax.legend(loc="upper right")
x_draw = np.linspace(0, 1, 1500)

def draw_frame(i):
    t, yy_up, yy_lw = frames[i]
    line_exact.set_data(x_draw, exact_solution(x_draw, t, "pulse"))
    line_up.set_data(x_anim, yy_up)
    line_lw.set_data(x_anim, yy_lw)
    ax.set_title(f"Перенос импульса: t={t:.2f}, C=0.8")
    return line_exact, line_up, line_lw

anim = FuncAnimation(fig, draw_frame, frames=len(frames), interval=85, blit=False)
with tempfile.TemporaryDirectory() as folder:
    gif_path = Path(folder) / "transport.gif"
    anim.save(str(gif_path), writer=PillowWriter(fps=12))
    gif_bytes = gif_path.read_bytes()
plt.close(fig)
display(Image(data=gif_bytes, format="gif"))
print(f"Анимация встроена в блокнот: {len(frames)} кадр(ов).")

## 6. Итоги

1. **Гладкая синусоида.** При $C=0.8$ обе схемы сходятся при увеличении $N$. Upwind имеет примерно первый порядок точности и заметное численное размытие. Лакс—Вендрофф имеет примерно второй порядок и лучше сохраняет форму синусоиды за пять оборотов.
2. **Сохранение массы.** На устойчивых расчётах сумма $M=\Delta x\sum_j u_j$ почти не меняется (расхождения порядка машинного округления). Но сохранение массы **не означает**, что концентрация по пространству вычислена правильно: upwind размывает прямоугольник, а Лакс—Вендрофф создаёт выбросы вокруг фронтов.
3. **Разрывный профиль.** При $C=0.5$ и $0.8$ upwind монотонен: для начальных значений от 0 до 1 новые экстремумы не возникают. Лакс—Вендрофф точнее передаёт положение резкого перехода, но имеет осцилляции; местами появляются $u<0$ и $u>1$, что не соответствует физическому смыслу концентрации.
4. **Случай $C=1$.** Подстановка $C=1$ в обе формулы даёт $u_j^{n+1}=u_{j-1}^n$. Поэтому профиль сдвигается ровно на одну ячейку за шаг — это особый случай без накопления ошибки переноса (с точностью до округления).
5. **Случай $C=1.1$.** Условие устойчивости нарушено. Малые высокочастотные возмущения многократно усиливаются; рост минимумов, максимумов и $E_1$ показывает, что такое решение нельзя использовать. Даже алгебраически консервативная схема при огромных амплитудах может терять численную точность вычисления массы из-за катастрофического округления.

**Ответ на вопрос задания:** нет, сохранение общего количества примеси — необходимая, но недостаточная проверка качества. Для физически осмысленного решения нужно одновременно контролировать ошибку, форму профиля, новые экстремумы и устойчивость по числу Куранта.